# Kedro Pipeline Runner

This notebook runs Kedro pipelines interactively within Jupyter.

**Benefits:**
- Run pipelines step by step
- Inspect intermediate outputs
- Debug pipeline issues
- Explore data between nodes

## Setup: Load Kedro Extension

In [ ]:
# Load Kedro IPython extension
%load_ext kedro.ipython

In [ ]:
# Reload Kedro project from current directory
%reload_kedro ./

## Check Available Pipelines

In [ ]:
# List all available pipelines
print("Available pipelines:")
for pipeline_name in context.pipelines.keys():
    print(f"  - {pipeline_name}")

## Pipeline 00: Data Exploration

This pipeline loads all CDCR mention data and creates a unified Parquet file.

**What it does:**
1. Scans for all `*_mentions.json` files
2. Adds metadata (dataset_name, split, entity_style)
3. Combines into single DataFrame
4. Saves to Parquet format (~15 MB)
5. Generates statistics and summary table

**Expected output:**
- `data/02_intermediate/all_mentions.parquet` (~15 MB)
- `data/02_intermediate/dataset_statistics.json` (~5 KB)
- `data/02_intermediate/summary_table.csv` (~2 KB)

In [ ]:
# Run the 00_data_exploration pipeline
session.run(pipeline_name="00_data_exploration")

## Inspect Pipeline Outputs

In [ ]:
# Load the unified mentions DataFrame
import pandas as pd

df = pd.read_parquet("data/02_intermediate/all_mentions.parquet")

print(f"DataFrame shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nDatasets: {df['dataset_name'].unique()}")
print(f"Splits: {df['split'].unique()}")
print(f"Entity styles: {df['entity_style'].value_counts().to_dict()}")

In [ ]:
# Display first few rows
df.head()

In [ ]:
# View dataset statistics
import json

with open("data/02_intermediate/dataset_statistics.json") as f:
    stats = json.load(f)

print("Dataset Statistics:")
print(f"  Total mentions: {stats['total_mentions']:,}")
print(f"  Total files: {stats['total_files']}")
print(f"\nBy entity style:")
for style, count in stats['by_entity_style'].items():
    print(f"  {style}: {count:,}")

if 'cluster_stats' in stats and stats['cluster_stats']:
    print(f"\nCluster statistics:")
    print(f"  Total clusters: {stats['cluster_stats']['total_clusters']:,}")
    print(f"  Singletons: {stats['cluster_stats']['singletons']:,}")
    print(f"  Non-singletons: {stats['cluster_stats']['non_singletons']:,}")
    print(f"  Avg cluster size: {stats['cluster_stats']['avg_cluster_size']:.2f}")
    print(f"  Max cluster size: {stats['cluster_stats']['max_cluster_size']}")
    print(f"  Estimated pairs: ~{stats['cluster_stats']['estimated_total_pairs']:,}")

In [ ]:
# View summary table
summary = pd.read_csv("data/02_intermediate/summary_table.csv")
print("\nSummary Table:")
summary

## Explore the Data

In [ ]:
# Mentions per dataset
print("Mentions per dataset:")
df.groupby('dataset_name').size().sort_values(ascending=False)

In [ ]:
# Mentions per split
print("Mentions per split:")
df.groupby('split').size()

In [ ]:
# Entity vs Event distribution per dataset
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

# Create pivot table
pivot = df.groupby(['dataset_name', 'entity_style']).size().unstack(fill_value=0)

# Plot
ax = pivot.plot(kind='bar', stacked=False, color=['steelblue', 'coral'], alpha=0.8)
ax.set_xlabel('Dataset', fontsize=12, fontweight='bold')
ax.set_ylabel('Number of Mentions', fontsize=12, fontweight='bold')
ax.set_title('Entity vs Event Mentions per Dataset', fontsize=14, fontweight='bold')
ax.legend(title='Entity Style', fontsize=10)
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Topics per dataset
print("Number of topics per dataset:")
df.groupby('dataset_name')['topic'].nunique().sort_values(ascending=False)

In [ ]:
# Example: View topics from a specific dataset
dataset = "HyperCoref"  # Change this to any dataset

topics = df[df['dataset_name'] == dataset]['topic'].unique()
print(f"\nTopics in {dataset} ({len(topics)} topics):")
print(sorted(topics)[:20])  # Show first 20
if len(topics) > 20:
    print(f"... and {len(topics) - 20} more")

In [ ]:
# Example: Filter data for a specific dataset and split
dataset = "ECBplus"
split = "train"

filtered = df[(df['dataset_name'] == dataset) & (df['split'] == split)]
print(f"\n{dataset} - {split}:")
print(f"  Total mentions: {len(filtered):,}")
print(f"  Entity mentions: {(filtered['entity_style'] == 'entity').sum():,}")
print(f"  Event mentions: {(filtered['entity_style'] == 'event').sum():,}")
print(f"  Topics: {filtered['topic'].nunique()}")

# Show sample mentions
print(f"\nSample mentions:")
filtered[['mention_id', 'tokens_str', 'topic', 'entity_style']].head(10)

## Advanced Analysis: Cluster Sizes

In [ ]:
# Analyze cluster sizes across all datasets
if 'coref_chain' in df.columns:
    cluster_sizes = df.groupby(['dataset_name', 'split', 'coref_chain']).size()
    
    print("Cluster Size Statistics:")
    print(cluster_sizes.describe())
    
    # Plot distribution
    fig, ax = plt.subplots(figsize=(12, 6))
    cluster_sizes.hist(bins=50, ax=ax, color='purple', alpha=0.7, edgecolor='black')
    ax.set_xlabel('Cluster Size', fontsize=11, fontweight='bold')
    ax.set_ylabel('Frequency', fontsize=11, fontweight='bold')
    ax.set_title('Distribution of Cluster Sizes', fontsize=12, fontweight='bold')
    ax.axvline(cluster_sizes.mean(), color='red', linestyle='--', linewidth=2, 
               label=f'Mean: {cluster_sizes.mean():.1f}')
    ax.axvline(cluster_sizes.median(), color='green', linestyle='--', linewidth=2,
               label=f'Median: {cluster_sizes.median():.0f}')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Show largest clusters
    print("\nTop 10 Largest Clusters:")
    print(cluster_sizes.nlargest(10))
else:
    print("No coref_chain column found")

## Run Other Pipelines

Use the cell below to run other pipelines as they are added.

## Pipeline 02: Pairs Creation

This pipeline prepares filtered train mentions for pair generation.

**What it does:**
1. Loads filtered train mentions (is_filtered=True, split='train')
2. Selects only required columns (10 columns instead of 30+)
3. Groups by dataset → entity_style → topic
4. Computes statistics for each group
5. Creates summary table

**Expected output:**
- `data/03_primary/mentions_for_pairs.parquet` - Train mentions with required columns
- `data/03_primary/grouping_statistics.json` - Hierarchical statistics
- `data/03_primary/mention_groups_summary.csv` - Summary table

In [ ]:
# Example: Run a specific pipeline
# session.run(pipeline_name="<pipeline_name>")

# Example: Run with specific tags
# session.run(pipeline_name="00_data_exploration", tags=["load", "statistics"])

# Example: Run a node by name
# session.run(node_names=["load_all_mentions_node"])

## Access Catalog Data

You can access data from the catalog directly using the `catalog` object.

In [ ]:
# List all datasets in catalog
print("Available datasets in catalog:")
for dataset_name in catalog.list():
    print(f"  - {dataset_name}")

In [ ]:
# Load data from catalog
# Example: Load the unified mentions DataFrame
# df_catalog = catalog.load("all_mentions_df")
# print(df_catalog.shape)

## Useful Commands

```python
# Reload Kedro project (if you changed code)
%reload_kedro ./

# Run full pipeline
session.run(pipeline_name="00_data_exploration")

# Run with specific tags
session.run(pipeline_name="00_data_exploration", tags=["load"])

# Run specific nodes
session.run(node_names=["load_all_mentions_node"])

# Access context
context.pipelines  # Available pipelines
context.params     # Parameters

# Access catalog
catalog.list()              # List all datasets
catalog.load("dataset_name")  # Load a dataset
catalog.save("dataset_name", data)  # Save data
```